In [1]:
import xml.etree.ElementTree as ET
from xml.dom import minidom
from pathlib import Path

In [2]:
ROOT_PATH = Path.cwd()
DATA_BASE_PATH = ROOT_PATH / "data"
VERTEX_FILE_PATH = DATA_BASE_PATH / "complete-osm-map" / "vertex.txt"
EDGES_FILE_PATH = DATA_BASE_PATH / "complete-osm-map" / "edges.txt"
OSM_OUTPUT_FILE_PATH = DATA_BASE_PATH / "melbourne_reconstructed.osm.xml"
GRAPH_OUTPUT_FILE_PATH = DATA_BASE_PATH / "melbourne_reconstructed.graphml"

In [3]:
import osmnx as ox
from datetime import datetime


def create_osm_from_txt(vertex_file: Path, edges_file: Path, output_file: Path):
    current_timestamp = (
        datetime.now().isoformat(timespec="seconds") + "Z"
    )  # Formato ISO 8601 com 'Z'
    default_version = "1"
    default_changeset = "1"
    default_uid = "1"
    default_user = "reconstructed_map"

    # Cria o elemento raiz do OSM
    osm = ET.Element("osm", version="0.6", generator="ReconstructionScript")

    # Dicionário para checar existência de nós (debug)
    nodes_processed = set()

    print("1. Processando Vértices...")
    with open(vertex_file, "r") as f:
        # Pular cabeçalhos se houver (ajuste conforme o arquivo real)
        # O formato diz que a primeira linha é count, vamos pular
        next(f)

        for line in f:
            parts = line.strip().split()
            # Formato esperado: [vertex's ID] [osm ID] [lon] [lat]
            if len(parts) < 4:
                continue

            v_id = parts[0]
            # osm_id_ref = parts[1] # Ignoramos o OSM ID real para manter consistência interna
            lon = parts[2]
            lat = parts[3]

            ET.SubElement(
                osm,
                "node",
                id=str(v_id),
                lat=str(lat),
                lon=str(lon),
                version=default_version,
                timestamp=current_timestamp,
                changeset=default_changeset,
                uid=default_uid,
                user=default_user,
                visible="true",
            )

            nodes_processed.add(v_id)

    print("2. Processando Arestas (Streets)...")
    with open(edges_file, "r") as f:
        # Pular cabeçalhos
        next(f)

        for line in f:
            parts = line.strip().split()
            # Formato esperado edges.txt: [edge ID] [start] [end] [dist]
            # Formato esperado streets.txt (mais completo):
            # [edge ID] [start] [lon] [lat] [end] [lon] [lat] [dist] [type] ...

            # Vamos assumir o formato streets.txt que é mais rico, mas ajuste indices se for edges.txt
            if len(parts) < 3:
                continue

            edge_id = parts[0]
            start_node = parts[1]
            # No streets.txt o end_node pode estar no índice 4 ou 5 dependendo das colunas de lat/lon
            # Ajuste este índice baseado no seu arquivo real.
            # Se for edges.txt simples: start=1, end=2
            end_node = parts[2]  # Assumindo edges.txt simples para o exemplo

            # Criar a Way
            way = ET.SubElement(
                osm,
                "way",
                id=str(edge_id),
                version=default_version,
                timestamp=current_timestamp,
                changeset=default_changeset,
                uid=default_uid,
                user=default_user,
                visible="true",
            )

            # Adicionar referências aos nós (Topologia)
            nd_start = ET.SubElement(way, "nd")
            nd_start.set("ref", start_node)

            nd_end = ET.SubElement(way, "nd")
            nd_end.set("ref", end_node)

            # Adicionar Tags para ser roteável
            # Se tiver coluna de tipo, use-a. Caso contrário, generalize.
            tag_hw = ET.SubElement(way, "tag")
            tag_hw.set("k", "highway")
            tag_hw.set("v", "secondary")  # Valor default seguro

            tag_ow = ET.SubElement(way, "tag")
            tag_ow.set("k", "oneway")
            tag_ow.set(
                "v", "yes"
            )  # Grafos acadêmicos geralmente são direcionados (digraph)

    print("3. Salvando arquivo OSM XML...")
    xml_str = minidom.parseString(ET.tostring(osm)).toprettyxml(indent="  ")

    output_file.parent.mkdir(parents=True, exist_ok=True)
    output_file.write_text(xml_str, encoding="utf-8")

    print(f"Sucesso! Arquivo salvo em: {output_file}")

    print("4. Convertendo OSM para GraphML...")
    G = ox.graph_from_xml(OSM_OUTPUT_FILE_PATH)
    ox.save_graphml(G, GRAPH_OUTPUT_FILE_PATH)
    print(f"GraphML salvo em: {GRAPH_OUTPUT_FILE_PATH}")


In [4]:
create_osm_from_txt(VERTEX_FILE_PATH, EDGES_FILE_PATH, OSM_OUTPUT_FILE_PATH)

1. Processando Vértices...
2. Processando Arestas (Streets)...
3. Salvando arquivo OSM XML...
Sucesso! Arquivo salvo em: /home/jose_edsouza/Documentos/Faculdade/TCC/repo/map-matching/datasets/melbourne/data/melbourne_reconstructed.osm.xml
4. Convertendo OSM para GraphML...
GraphML salvo em: /home/jose_edsouza/Documentos/Faculdade/TCC/repo/map-matching/datasets/melbourne/data/melbourne_reconstructed.graphml
